In [3]:
# Imports
import numpy as np
import os as os
import netCDF4 as nc
import xarray as xr

In [4]:
# Open the file
nc_ERA5 = nc.Dataset('/home/chinahg/GCresearch/contrailuncertainty/ERA5dataset.nc', 'r', format='NETCDF4_CLASSIC')

# Get useful values
# pressures = nc_ERA5.variables['level'] # km
# print(pressures.shape)
# RHs = nc_ERA5.variables['r'][:]*100 # Percent

temperatures = nc_ERA5.variables['t']
DAL746_temp = temperatures[0,2,5,13] # time, level, latitude, longitude
DAL746_RH = nc_ERA5.variables['r'][0,2,5,13]

UPS942_temp = temperatures[1,1,7,40] # time, level, latitude, longitude
UPS942_RH = nc_ERA5.variables['r'][1,1,7,40]

FDX1026_temp = temperatures[0,2,0,35] # time, level, latitude, longitude
FDX1026_RH = nc_ERA5.variables['r'][0,2,0,35]

UAL1947_temp = temperatures[0,1,7,19] # time, level, latitude, longitude
UAL1947_RH = nc_ERA5.variables['r'][0,1,7,19]

AAL305_temp = temperatures[0,2,4,20] # time, level, latitude, longitude
AAL305_RH = nc_ERA5.variables['r'][0,2,4,20]

In [5]:
ds = xr.open_dataset("/home/chinahg/GCresearch/contrailuncertainty/ERA5dataset.nc")
ds

<xarray.Dataset>
Dimensions:    (longitude: 41, latitude: 9, level: 3, time: 2)
Coordinates:
  * longitude  (longitude) float32 -105.0 -104.8 -104.5 ... -95.5 -95.25 -95.0
  * latitude   (latitude) float32 40.0 39.75 39.5 39.25 ... 38.5 38.25 38.0
  * level      (level) int32 200 225 250
  * time       (time) datetime64[ns] 2022-04-21T08:00:00 2022-04-21T09:00:00
Data variables:
    r          (time, level, latitude, longitude) float32 ...
    q          (time, level, latitude, longitude) float32 ...
    t          (time, level, latitude, longitude) float32 ...
    u          (time, level, latitude, longitude) float32 ...
    v          (time, level, latitude, longitude) float32 ...
    w          (time, level, latitude, longitude) float32 ...
    vo         (time, level, latitude, longitude) float32 ...
Attributes:
    Conventions:  CF-1.6
    history:      2023-08-03 17:35:13 GMT by grib_to_netcdf-2.25.1: /opt/ecmw...

In [9]:
met = xr.open_dataset("/home/chinahg/GCresearch/APCEMM/examples/Example3_met_input/example_met_file.nc")
met

<xarray.Dataset>
Dimensions:                  (altitude: 37, time: 24)
Coordinates:
  * altitude                 (altitude) float64 0.07068 0.2804 ... 42.26 47.59
  * time                     (time) float64 15.0 16.0 17.0 ... 12.0 13.0 14.0
Data variables: (12/31)
    var_length               float64 ...
    pressure                 (altitude) float64 ...
    temperature              (altitude, time) float64 ...
    relative_humidity        (altitude) float64 ...
    relative_humidity_ice    (altitude) float64 ...
    shear                    (altitude, time) float64 ...
    ...                       ...
    datetime_takeoff_month   float64 ...
    datetime_takeoff_day     float64 ...
    datetime_takeoff_hour    float64 ...
    datetime_takeoff_minute  float64 ...
    emit_depth               float64 ...
    saturation_depth         float64 ...

In [8]:
# modify met data for appropriate flights
# Assuming only temperature and shear are time dependent
# RH is only altitude dependent 

# Importing the modules
import os
import shutil
import igra
import matplotlib.pyplot as plt
import numpy as np

# Import IGRA data
stations = igra.download.stationlist('/tmp')
station_num = "USM00072469"
igra.download.station(station_num, "/tmp")
data, station = igra.read.igra(station_num, "/tmp/"+station_num+"-data.txt.zip")
data2022 = data.rhumi.loc["2022-01-01":"2022-07-09"] # RH for every day and every pressure level
RH20220101 = data2022.loc["2022-01-01T00:00:00.000000000"]

# DAL746
flight_name = 'DAL746'

for i in range(245): # iterate through days in the year

    # Create new met file to edit with RH and temp data from radiosonde data
    date = str(i) +'_2022'
    met_name = date + flight_name

    dest_dir = '/home/chinahg/GCresearch/contrailuncertainty/APCEMM_results'
    src_file = '/home/chinahg/GCresearch/APCEMM/examples/Example3_met_input/example_met_file.nc'
    shutil.copy(src_file,dest_dir) #copy the file to destination dir

    dst_file = os.path.join(dest_dir,'example_met_file.nc')
    new_dst_file_name = os.path.join(dest_dir, met_name + 'met')
    os.rename(dst_file, new_dst_file_name) #rename

    # Edit nc file with RH profiles
    met = xr.open_dataset(dst_file)
    # need to convert pressure to altitude for RH profiles (radiosonde --> netCDF met)

    # Then, run APCEMM with modified netCDF input file (see example 3)
    

/home/chinahg/GCresearch/contrailuncertainty/APCEMM_results/example_met_file.nc
/home/chinahg/GCresearch/contrailuncertainty/APCEMM_results/01_2022DAL746met
